# Topology Summary Generator

This notebook provides tools to generate and inject descriptive summaries into topology YAML files.

## Features:
- 📊 Generate detailed topology summaries
- 🎨 Create ASCII art diagrams
- 💾 Inject summaries into YAML files as comment headers
- 🔒 Automatic backup creation
- 📄 Works with model-based topologies
- 🔌 Supports facility ports

## Setup

In [ ]:
import sys
from pathlib import Path

# Configure paths
repo_root = Path.cwd().parent.parent

print(f"✅ Python path configured")
print(f"   Repository root: {repo_root}")

In [ ]:
# Import modules from installed package
from fabric_generic_cluster import (
    load_topology_from_yaml_file,
    SiteTopology
)

from fabric_generic_cluster import topology_viewer as viewer

print("✅ Modules imported successfully")

In [ ]:
# Define YAML directory
YAML_DIR = repo_root / "model"

# Topology YAML file path
yaml_file = YAML_DIR / "_slice_topology_1.yml"

print(f"✅ YAML directory: {YAML_DIR}")

# Check if file exists
if yaml_file.exists():
    print(f"✅ Found topology file: {yaml_file.name}")
else:
    print(f"❌ File not found: {yaml_file}")
    print(f"\nAvailable YAML files in {YAML_DIR}:")
    for f in YAML_DIR.glob("*.yml"):
        print(f"  - {f.name}")

## Load Topology

Load and validate the topology:

In [ ]:
try:
    topology = load_topology_from_yaml_file(str(yaml_file))
    print("✅ Topology loaded and validated successfully!")
    print(f"\n📊 Statistics:")
    print(f"   • Nodes: {len(topology.site_topology_nodes.nodes)}")
    print(f"   • Networks: {len(topology.site_topology_networks.networks)}")
    
    # Check for facility ports
    if topology.has_facility_ports():
        print(f"   • Facility Ports: {len(topology.site_topology_facility_ports.facility_ports)}")
    
    # List nodes
    print(f"\n🖥️ Nodes:")
    for node in topology.site_topology_nodes.iter_nodes():
        worker_info = f" [worker: {node.worker}]" if node.has_worker_constraint() else ""
        postboot_info = " [has postboot]" if node.specific.has_postboot_commands() else ""
        print(f"   • {node.hostname} ({node.site}){worker_info}{postboot_info}")
    
    # List networks
    print(f"\n🌐 Networks:")
    for network in topology.site_topology_networks.iter_networks():
        subnet_info = network.ipv4_subnet or network.ipv6_subnet or "auto-assigned"
        print(f"   • {network.name} ({network.type}) - {subnet_info}")
    
    # List facility ports if any
    if topology.has_facility_ports():
        print(f"\n🔌 Facility Ports:")
        for fp in topology.site_topology_facility_ports.iter_facility_ports():
            print(f"   • {fp.name} @ {fp.site} (VLAN {fp.vlan}) → {fp.binding}")
        
except Exception as e:
    print(f"❌ Error loading topology: {e}")
    import traceback
    traceback.print_exc()

---

# Option 1: Command-Line Tool

Use the `fabric-topology-summary` command-line tool.

## Basic Usage

In [ ]:
# Display command-line usage examples
print("""
Command-Line Usage Examples:
═══════════════════════════════════════════════════════════════

1. Inject summary into YAML file (creates backup automatically):
   fabric-topology-summary topology.yml

2. Generate summary to separate file:
   fabric-topology-summary topology.yml --output summary.yml

3. Dry run (preview without modifying file):
   fabric-topology-summary topology.yml --dry-run

4. Include ASCII diagram:
   fabric-topology-summary topology.yml --ascii

5. Get help:
   fabric-topology-summary --help

""")

## Run from Notebook

In [ ]:
# Display help
!fabric-topology-summary --help

In [ ]:
# Preview summary without modifying file
!fabric-topology-summary {yaml_file} --dry-run

In [ ]:
# Generate summary to separate file (non-destructive)
output_file = YAML_DIR / "topology_summary.yml"
!fabric-topology-summary {yaml_file} --output {output_file}

# Display the first 50 lines
if output_file.exists():
    print("\n" + "="*70)
    print("Generated Summary (first 50 lines):")
    print("="*70)
    lines = output_file.read_text().split('\n')[:50]
    print('\n'.join(lines))
    if len(output_file.read_text().split('\n')) > 50:
        print("...\n(file continues)")

---

# Option 2: Programmatic API

Use Python functions for more control and integration with your workflow.

## 2.1 Preview Summary (Non-Destructive)

In [ ]:
# Generate summary text with ASCII diagram
summary = viewer.generate_yaml_summary(topology, include_ascii=True)

print("="*70)
print("GENERATED SUMMARY PREVIEW")
print("="*70)
print(summary)

## 2.2 Generate Summary Without ASCII Diagram

In [ ]:
# Generate summary without ASCII diagram
summary_no_ascii = viewer.generate_yaml_summary(topology, include_ascii=False)

print("="*70)
print("SUMMARY WITHOUT ASCII DIAGRAM")
print("="*70)
print(summary_no_ascii)

## 2.3 Save Summary to Separate File

In [ ]:
# Save summary to a separate text file
output_file = YAML_DIR / "topology_summary_detailed.txt"

summary = viewer.generate_yaml_summary(topology, include_ascii=True)
output_file.write_text(summary)

print(f"✅ Summary saved to: {output_file}")
print(f"   File size: {output_file.stat().st_size:,} bytes")
print(f"   Lines: {len(summary.split(chr(10)))}")

## 2.4 Inject Summary into YAML File

⚠️ **Warning**: This will modify your YAML file (backup is created automatically)

In [ ]:
# Configuration
create_backup = True          # Create .bak file before modifying
include_ascii_diagram = True  # Include ASCII network diagram

print(f"Configuration:")
print(f"  • Target file: {yaml_file}")
print(f"  • Create backup: {create_backup}")
print(f"  • Include ASCII: {include_ascii_diagram}")
print("\nℹ️  A backup file (.bak) will be created before modification.")
print("\n⚠️  Run the next cell to inject the summary into the YAML file.")

In [ ]:
# Inject summary into YAML file
try:
    viewer.inject_summary_into_yaml_file(
        str(yaml_file),
        topology,
        include_ascii=include_ascii_diagram,
        backup=create_backup
    )
    
    print("\n✅ Summary successfully injected into YAML file!")
    
    if create_backup:
        backup_file = yaml_file.with_suffix(yaml_file.suffix + '.bak')
        if backup_file.exists():
            print(f"\n💾 Backup saved to: {backup_file}")
    
    # Display first few lines of updated file
    print("\n📄 First 40 lines of updated file:")
    print("="*70)
    lines = yaml_file.read_text().split('\n')[:40]
    print('\n'.join(lines))
    print("...")
    
except Exception as e:
    print(f"❌ Error injecting summary: {e}")
    import traceback
    traceback.print_exc()

## 2.5 Restore from Backup (If Needed)

In [ ]:
# Check for backup file
backup_file = yaml_file.with_suffix(yaml_file.suffix + '.bak')

if backup_file.exists():
    print(f"✅ Found backup: {backup_file}")
    print(f"   Size: {backup_file.stat().st_size:,} bytes")
    print("\n⚠️  Run the next cell to restore from backup.")
else:
    print(f"❌ No backup file found: {backup_file}")

In [ ]:
# Uncomment and run to restore from backup
# backup_file = yaml_file.with_suffix(yaml_file.suffix + '.bak')
# if backup_file.exists():
#     backup_content = backup_file.read_text()
#     yaml_file.write_text(backup_content)
#     print(f"✅ File restored from backup: {yaml_file}")
# else:
#     print(f"❌ Backup file not found")

---

# Additional Visualization Tools

## Display Detailed Text Summary

In [ ]:
# Print detailed topology summary
viewer.print_topology_summary(topology)

## Display Compact Summary

In [ ]:
# Print compact table-style summary
viewer.print_compact_summary(topology)

## Display ASCII Topology Diagram

In [ ]:
# Print ASCII art representation
viewer.print_ascii_topology(topology)

## Display Network Details

In [ ]:
# Show details for a specific network
network_name = "net_local_1"  # Change this to your network name

viewer.print_network_details(topology, network_name)

## Generate Topology Graph (Visual)

In [ ]:
# Draw topology as a graph (requires matplotlib)
try:
    viewer.draw_topology_graph(
        topology,
        figsize=(14, 10),
        show_ip=True,
        save_path=str(YAML_DIR / "topology_graph.png")
    )
except Exception as e:
    print(f"⚠️  Could not generate graph: {e}")
    print("   Make sure matplotlib and networkx are installed.")

---

# File Comparison Tools

## Compare Original vs Updated File

In [ ]:
backup_file = yaml_file.with_suffix(yaml_file.suffix + '.bak')

if backup_file.exists():
    original_size = backup_file.stat().st_size
    updated_size = yaml_file.stat().st_size
    
    print("📊 File Comparison:")
    print(f"   Original: {original_size:,} bytes")
    print(f"   Updated:  {updated_size:,} bytes")
    print(f"   Difference: +{updated_size - original_size:,} bytes")
    
    # Count lines and comments
    original_lines = backup_file.read_text().split('\n')
    updated_lines = yaml_file.read_text().split('\n')
    
    original_comments = sum(1 for line in original_lines if line.strip().startswith('#'))
    updated_comments = sum(1 for line in updated_lines if line.strip().startswith('#'))
    
    print(f"\n   Original lines: {len(original_lines)}")
    print(f"   Updated lines:  {len(updated_lines)}")
    print(f"   Lines added:    +{len(updated_lines) - len(original_lines)}")
    
    print(f"\n   Original comment lines: {original_comments}")
    print(f"   Updated comment lines:  {updated_comments}")
    print(f"   New comments added:     +{updated_comments - original_comments}")
else:
    print("❌ No backup file available for comparison")

## Batch Process Multiple Files

In [ ]:
# Process multiple topology files
yaml_files_to_process = [
    YAML_DIR / "_slice_topology.yml",
    YAML_DIR / "_slice_topology_2.yml",
    YAML_DIR / "_slice_topology_3.yml",
    # Add more files here as needed
]

print("📦 Batch Processing Topology Files\n")

results = {"success": 0, "skipped": 0, "failed": 0}

for yaml_file_path in yaml_files_to_process:
    if not yaml_file_path.exists():
        print(f"⏭️  Skipping (not found): {yaml_file_path.name}")
        results["skipped"] += 1
        continue
    
    try:
        print(f"\n📄 Processing: {yaml_file_path.name}")
        
        # Load topology
        topo = load_topology_from_yaml_file(str(yaml_file_path))
        
        # Inject summary
        viewer.inject_summary_into_yaml_file(
            str(yaml_file_path),
            topo,
            include_ascii=True,
            backup=True
        )
        
        print(f"   ✅ Summary injected")
        results["success"] += 1
        
    except Exception as e:
        print(f"   ❌ Error: {e}")
        results["failed"] += 1

print("\n" + "="*70)
print("Batch Processing Results:")
print(f"   ✅ Successful: {results['success']}")
print(f"   ⏭️  Skipped:    {results['skipped']}")
print(f"   ❌ Failed:     {results['failed']}")
print("="*70)

---

# Export Tools

## Export to JSON

In [ ]:
# Export topology to JSON format
json_output = YAML_DIR / "topology_export.json"

try:
    viewer.export_topology_to_json(topology, str(json_output))
    print(f"✅ Topology exported to JSON: {json_output}")
except Exception as e:
    print(f"❌ Export failed: {e}")

## Export to Graphviz DOT

In [ ]:
# Export topology to Graphviz DOT format
dot_output = YAML_DIR / "topology_export.dot"

try:
    viewer.export_topology_to_dot(topology, str(dot_output))
    print(f"✅ Topology exported to DOT: {dot_output}")
    print(f"\n💡 Visualize with: dot -Tpng {dot_output} -o topology.png")
except Exception as e:
    print(f"❌ Export failed: {e}")

---

# Summary

This notebook provides two ways to generate topology summaries:

## Option 1: Command-Line Tool
- ✅ Simple one-line commands using `fabric-topology-summary`
- ✅ Easy to automate in scripts
- ✅ Good for CI/CD pipelines
- ✅ Installed with the package

## Option 2: Programmatic API
- ✅ Full control from Python/Jupyter
- ✅ Preview before modifying
- ✅ Integrate with your workflow
- ✅ Batch processing support
- ✅ Multiple visualization options

Both approaches:
- 🔒 Create automatic backups
- 📊 Generate detailed summaries
- 🎨 Include ASCII diagrams
- ✅ Work with model-based topologies
- 🔌 Support facility ports
- 📍 Display worker constraints
- 🔧 Show postboot commands
